In [20]:
import os, glob, subprocess, shlex
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, log_loss
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
import optuna
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import joblib
import warnings
import torch

warnings.filterwarnings("ignore")
RANDOM_SEED = 42

In [21]:
def detect_gpu():
    try:
        has_torch_gpu = torch.cuda.is_available()
        print(f"detect_gpu: PyTorch detected. cuda_available={has_torch_gpu}, device_count={torch.cuda.device_count()}")
        if has_torch_gpu:
            return True
    except Exception:
        pass
        
USE_GPU = detect_gpu()

detect_gpu: PyTorch detected. cuda_available=True, device_count=1


In [192]:
def load_csvs(folder_path, patterns):
    all_dfs = []
    for pat in patterns:
        for path in glob.glob(os.path.join(folder_path, pat)):
            df = pd.read_csv(path)
            df["source_file"] = os.path.basename(path)
            df = df.sort_values("frame").iloc[int(len(df)*0.2) : int(len(df)*0.8) : 10]
            all_dfs.append(df)
            print("Loaded:", path, "shape:", df.shape, end="\r")
    if not all_dfs:
        raise ValueError("No files loaded. Check folder_path/patterns.")
    
    final_df = pd.concat(all_dfs, ignore_index=True)
    print("Combined shape:", final_df.shape)
    return final_df

def load_all_csvs(folder_path, patterns):
    all_dfs = []
    for pat in patterns:
        for path in glob.glob(os.path.join(folder_path, pat)):
            df = pd.read_csv(path)
            df["source_file"] = os.path.basename(path)
            all_dfs.append(df)
            print("Loaded:", path, "shape:", df.shape, end="\r")
    if not all_dfs:
        raise ValueError("No files loaded. Check folder_path/patterns.")
    
    final_df = pd.concat(all_dfs, ignore_index=True)
    print("Combined shape:", final_df.shape)
    return final_df


def assign_label(filename: str) -> int:
    if "_correct_" in filename or "_cor_" in filename:
        return 1
    elif "_incorrect_" in filename:
        return 0
    return np.nan

In [23]:
def clean_missing_coords(df, pts=33, drop_all_zero=True):
    """
    - เติม z{i} เป็น 0 ถ้าไม่มี
    - แปลง inf -> NaN แล้ว drop rows ที่มี NaN ใน x{i}, y{i}
    - ถ้าต้องการ ลบ rows ที่ทุก coordinate เป็น 0
    """
    # เติม z columns ถ้าไม่มี
    for i in range(pts):
        if f"z{i}" not in df.columns:
            df[f"z{i}"] = 0.0

    # แปลง inf เป็น NaN
    df = df.replace([np.inf, -np.inf], np.nan)

    # ตรวจ column x,y มีครบไหม
    required_cols = []
    for i in range(pts):
        required_cols.append(f"x{i}")
        required_cols.append(f"y{i}")
    for c in required_cols:
        if c not in df.columns:
            raise ValueError(f"Missing required coordinate column: {c}")

    before = len(df)
    df_clean = df.dropna(subset=required_cols)
    after = len(df_clean)
    print(f"clean_missing_coords: dropped {before-after} rows with missing x/y coords (from {before} -> {after})")

    # ลบ rows ที่ทุก coordinate เป็น 0 (option)
    if drop_all_zero:
        coord_cols = required_cols + [f"z{i}" for i in range(pts)]
        all_zero_mask = (df_clean[coord_cols] == 0).all(axis=1)
        nz_before = len(df_clean)
        if all_zero_mask.any():
            df_clean = df_clean.loc[~all_zero_mask].reset_index(drop=True)
            nz_after = len(df_clean)
            print(f"clean_missing_coords: removed {nz_before-nz_after} all-zero rows (from {nz_before} -> {nz_after})")
    return df_clean.reset_index(drop=True)

In [116]:
def preprocess_and_features(df, pts=33):
    # filter label
    df['label'] = df['source_file'].apply(assign_label)
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)

    # CLEAN missing values
    df = clean_missing_coords(df, pts=pts, drop_all_zero=True)

    # ถอดฟีเจอร์ตำแหน่งเป็น numpy array per row
    xs, ys, zs = [], [], []
    for i in range(pts):
        xs.append(df[f"x{i}"].values)
        ys.append(df[f"y{i}"].values)
        zs.append(df[f"z{i}"].values if f"z{i}" in df.columns else np.zeros(len(df)))
    X_arr = np.vstack(xs).T  # shape (N, pts)
    Y_arr = np.vstack(ys).T
    Z_arr = np.vstack(zs).T

    # center: ใช้ mean ของ points เป็น center (ปรับได้)
    center_x = X_arr.mean(axis=1, keepdims=True)
    center_y = Y_arr.mean(axis=1, keepdims=True)
    center_z = Z_arr.mean(axis=1, keepdims=True)

    Xc = X_arr - center_x
    Yc = Y_arr - center_y
    Zc = Z_arr - center_z

    # scale
    torso_scale = np.linalg.norm(np.stack([Xc, Yc, Zc], axis=2), axis=2).mean(axis=1, keepdims=True)
    torso_scale = np.where(torso_scale <= 1e-6, 1.0, torso_scale)

    Xs = Xc / torso_scale
    Ys = Yc / torso_scale
    Zs = Zc / torso_scale

    feat_coords = np.hstack([Xs, Ys, Zs])  # shape (N, pts*3)

    dist_to_center = np.linalg.norm(np.stack([Xs, Ys, Zs], axis=2), axis=2)  # (N, pts)
    dist_mean = dist_to_center.mean(axis=1, keepdims=True)
    dist_std  = dist_to_center.std(axis=1, keepdims=True)
    dist_max  = dist_to_center.max(axis=1, keepdims=True)

    # pairwise key indices (ปรับ mapping ให้ตรง dataset ของคุณ)
    key_indices = [11,12,23,24,13,14,15,16]
    pair_feats = []
    for i in range(len(key_indices)):
        for j in range(i+1, len(key_indices)):
            a = key_indices[i]
            b = key_indices[j]
            vec = np.stack([Xs[:,a]-Xs[:,b], Ys[:,a]-Ys[:,b], Zs[:,a]-Zs[:,b]], axis=1)
            d = np.linalg.norm(vec, axis=1, keepdims=True)
            pair_feats.append(d)
    pair_feats = np.hstack(pair_feats) if pair_feats else np.zeros((len(df), 1))

    def angle_between(a_idx, b_idx, c_idx):
        ba = np.stack([Xs[:,a_idx]-Xs[:,b_idx], Ys[:,a_idx]-Ys[:,b_idx], Zs[:,a_idx]-Zs[:,b_idx]], axis=1)
        bc = np.stack([Xs[:,c_idx]-Xs[:,b_idx], Ys[:,c_idx]-Ys[:,b_idx], Zs[:,c_idx]-Zs[:,b_idx]], axis=1)
        na = ba / (np.linalg.norm(ba, axis=1, keepdims=True) + 1e-8)
        nc = bc / (np.linalg.norm(bc, axis=1, keepdims=True) + 1e-8)
        cosang = (na * nc).sum(axis=1, keepdims=True)
        cosang = np.clip(cosang, -1, 1)
        ang = np.arccos(cosang)
        return ang

    angles = []
    try:
        angles.append(angle_between(11,13,15))  # left elbow
        angles.append(angle_between(12,14,16))  # right elbow
        angles.append(angle_between(23,11,13))  # left hip-shoulder-elbow
        angles.append(angle_between(24,12,14))  # right hip-shoulder-elbow
    except Exception:
        pass
    angles = np.hstack(angles) if angles else np.zeros((len(df),1))

    X_feat = np.hstack([feat_coords, dist_mean, dist_std, dist_max, pair_feats, angles])
    y = df['label'].values
    return X_feat, y, df

In [117]:
def prepare_X_y_for_tuning(X, y, impute_if_nan=True):
    """
    ตรวจหา NaN/inf ใน X, แปลง inf->nan, ถ้ามี NaN จะ impute ด้วย mean (หรือถ้าต้องการ drop ให้เปลี่ยน)
    คืนค่า X_clean (np.ndarray) และ y (np.ndarray)
    """
    X = np.asarray(X, dtype=float)
    # convert inf to nan
    X[np.isinf(X)] = np.nan
    n_nan = np.isnan(X).sum()
    print(f"prepare: X shape={X.shape}, total NaNs={n_nan}")
    if n_nan > 0:
        if impute_if_nan:
            imp = SimpleImputer(strategy='mean')
            X = imp.fit_transform(X)
            print("prepare: performed mean-imputation for NaNs")
        else:
            # drop rows with any NaN
            mask = ~np.isnan(X).any(axis=1)
            print(f"prepare: dropping {len(X) - mask.sum()} rows with NaNs")
            X = X[mask]
            y = np.asarray(y)[mask]
    return X, np.asarray(y)

def get_safe_n_splits(y, requested_folds=4):
    """
    เลือกจำนวน folds ให้ไม่เกินจำนวนตัวอย่างต่อคลาสที่น้อยที่สุด
    """
    cnt = Counter(y)
    min_cnt = min(cnt.values())
    n_splits = min(requested_folds, max(2, min_cnt))  # ต้อง >=2 เพื่อ CV
    if min_cnt < 2:
        raise ValueError(f"Not enough samples in at least one class for CV. class counts: {dict(cnt)}")
    print(f"CV: class_counts={dict(cnt)}, using n_splits={n_splits}")
    return n_splits

In [118]:
def objective_xgb(trial, X_raw, y_raw, requested_folds=4):
    X, y = prepare_X_y_for_tuning(X_raw, y_raw, impute_if_nan=True)
    try:
        n_splits = get_safe_n_splits(y, requested_folds)
    except ValueError as e:
        print("objective_xgb: insufficient class samples:", e)
        return 0.0  # very bad score but completes trial

    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 0.3),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.4, 1.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
        'random_state': RANDOM_SEED,
        'use_label_encoder': False
    }
    if USE_GPU:
        param.update({'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor', 'gpu_id': 0})
    else:
        param.update({'tree_method': 'hist'})

    clf = xgb.XGBClassifier(**param)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    try:
        scores = cross_val_score(clf, X, y, scoring='roc_auc', cv=cv, n_jobs=1)
        if np.any(np.isnan(scores)):
            print("objective_xgb: got NaN in CV scores:", scores)
            return 0.0
        return float(np.mean(scores))
    except Exception as e:
        print("objective_xgb: exception during cross_val_score:", e)
        return 0.0

def objective_lgb(trial, X_raw, y_raw, requested_folds=4):
    X, y = prepare_X_y_for_tuning(X_raw, y_raw, impute_if_nan=True)
    try:
        n_splits = get_safe_n_splits(y, requested_folds)
    except ValueError as e:
        print("objective_lgb: insufficient class samples:", e)
        return 0.0

    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
        'num_leaves': trial.suggest_int('num_leaves', 7, 512),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 0.3),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_uniform('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.4, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 10.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 10.0),
        'random_state': RANDOM_SEED
    }
    if USE_GPU:
        param.update({'device': 'gpu', 'gpu_platform_id': 0, 'gpu_device_id': 0})
    clf = lgb.LGBMClassifier(**param)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    try:
        scores = cross_val_score(clf, X, y, scoring='roc_auc', cv=cv, n_jobs=1)
        if np.any(np.isnan(scores)):
            print("objective_lgb: got NaN in CV scores:", scores)
            return 0.0
        return float(np.mean(scores))
    except Exception as e:
        print("objective_lgb: exception during cross_val_score:", e)
        return 0.0

In [27]:
def run_pipeline(folder_path, patterns, n_trials=40, model_choice='xgb'):
    df = load_csvs(folder_path, patterns)
    X, y, df = preprocess_and_features(df)
    print("Feature shape:", X.shape)

    # inspect class distribution
    print("Class distribution before tuning:", dict(Counter(y)))

    # standardize
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    # compute sample_weight
    classes = np.unique(y)
    class_w = compute_class_weight('balanced', classes=classes, y=y)
    sample_weight = np.array([class_w[cls] for cls in y])

    # Optuna tuning
    if model_choice == 'xgb':
        study = optuna.create_study(direction='maximize', study_name='xgb_study')
        func = lambda trial: objective_xgb(trial, Xs, y, requested_folds=4)
        study.optimize(func, n_trials=n_trials, show_progress_bar=True)
        completed = [t for t in study.trials if t.state.is_complete()]
        if len(completed) == 0:
            raise ValueError("No completed Optuna trials. Check your data, CV folds, or imputation strategy.")
        print("Best XGB trial:", study.best_trial.params, "AUC:", study.best_value)
        best_params = study.best_trial.params
        if USE_GPU:
            best_params.update({'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor', 'gpu_id': 0})
        else:
            best_params.update({'tree_method': 'hist'})
        best_params.update({'use_label_encoder': False, 'random_state': RANDOM_SEED})
        best_clf = xgb.XGBClassifier(**best_params)
    elif model_choice == 'lgb':
        study = optuna.create_study(direction='maximize', study_name='lgb_study')
        func = lambda trial: objective_lgb(trial, Xs, y, requested_folds=4)
        study.optimize(func, n_trials=n_trials, show_progress_bar=True)
        completed = [t for t in study.trials if t.state.is_complete()]
        if len(completed) == 0:
            raise ValueError("No completed Optuna trials. Check your data, CV folds, or imputation strategy.")
        print("Best LGB trial:", study.best_trial.params, "AUC:", study.best_value)
        best_params = study.best_trial.params
        if USE_GPU:
            best_params.update({'device': 'gpu', 'gpu_platform_id': 0, 'gpu_device_id': 0})
        best_params.update({'random_state': RANDOM_SEED})
        best_clf = lgb.LGBMClassifier(**best_params)
    elif model_choice == 'cat':
        if USE_GPU:
            best_clf = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, task_type='GPU', verbose=0, random_state=RANDOM_SEED)
        else:
            best_clf = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, task_type='CPU', verbose=0, random_state=RANDOM_SEED)
    else:
        raise ValueError("model_choice must be 'xgb','lgb' or 'cat'")

    # final split and fit
    X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(Xs, y, sample_weight, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

    if model_choice == 'xgb':
        try:
            best_clf.fit(X_tr, y_tr, sample_weight=w_tr,
                         eval_set=[(X_val, y_val)],
                         eval_metric='logloss',
                         early_stopping_rounds=50,
                         verbose=50)
        except Exception as e:
            print("Warning: XGBoost training with GPU failed or other error, retrying with CPU. Error:", e)
            best_clf.set_params(tree_method='hist', predictor='cpu_predictor')
            best_clf.fit(X_tr, y_tr, sample_weight=w_tr,
                         eval_set=[(X_val, y_val)],
                         eval_metric='logloss',
                         early_stopping_rounds=50,
                         verbose=50)
    elif model_choice == 'lgb':
        try:
            best_clf.fit(X_tr, y_tr, sample_weight=w_tr,
                         eval_set=[(X_val, y_val)],
                         eval_metric='binary_logloss',
                         early_stopping_rounds=50,
                         verbose=50)
        except Exception as e:
            print("Warning: LightGBM training failed with GPU flags, retrying CPU. Error:", e)
            params = best_clf.get_params()
            for k in ['device','gpu_platform_id','gpu_device_id']:
                if k in params: params.pop(k, None)
            best_clf = lgb.LGBMClassifier(**params)
            best_clf.fit(X_tr, y_tr, sample_weight=w_tr,
                         eval_set=[(X_val, y_val)],
                         eval_metric='binary_logloss',
                         early_stopping_rounds=50,
                         verbose=50)
    else:
        best_clf.fit(X_tr, y_tr, sample_weight=w_tr)

    # evaluate
    if hasattr(best_clf, "predict_proba"):
        probs = best_clf.predict_proba(X_val)
    else:
        # fallback to xgboost Booster predict
        try:
            dval = xgb.DMatrix(X_val)
            probs = best_clf.predict(dval)
        except Exception:
            raise RuntimeError("Model does not support predict_proba and is not an xgboost.Booster.")

    n_classes = len(np.unique(y))
    if n_classes == 2:
        if probs.ndim > 1:
            y_prob = probs[:,1]
        else:
            y_prob = probs
        y_pred = (y_prob >= 0.5).astype(int)
        auc = roc_auc_score(y_val, y_prob)
        ll = log_loss(y_val, y_prob)
    else:
        y_prob = probs
        y_pred = probs.argmax(axis=1)
        auc = roc_auc_score(y_val, y_prob, multi_class="ovo")
        ll = log_loss(y_val, y_prob)

    acc = accuracy_score(y_val, y_pred)
    print("Validation AUC:", auc)
    print("Validation Accuracy:", acc)
    print("Validation LogLoss:", ll)
    print("Classification Report:\n", classification_report(y_val, y_pred, digits=4))
    print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))

    # save scaler+model
    joblib.dump({'scaler': scaler, 'model': best_clf}, f"best_model_{model_choice}.joblib")
    print("Saved best_model_%s.joblib" % model_choice)
    return best_clf, scaler, Xs, y

In [28]:
def run_pipeline(folder_path, patterns, n_trials=40, model_choice='xgb'):
    df = load_csvs(folder_path, patterns)
    X, y, df = preprocess_and_features(df)
    print("Feature shape:", X.shape)

    # inspect class distribution
    print("Class distribution before tuning:", dict(Counter(y)))

    classes = np.unique(y)
    class_w = compute_class_weight('balanced', classes=classes, y=y)
    sample_weight = np.array([class_w[cls] for cls in y])


In [ ]:
if __name__ == "__main__":
    folder_path = r"../data_sample/csv_output"   # ปรับเป็น path ที่แท้จริง
    patterns = ["*_bicep_*.csv", "*any*.csv"]
    # ลอง XGBoost (จะใช้ GPU ถ้า detect เจอ)
    model, scaler, Xs, y = run_pipeline(folder_path, patterns, n_trials=30, model_choice='xgb')
    # ถ้าต้องการลอง LightGBM: model_choice='lgb'
    # ถ้าต้องการ CatBoost: model_choice='cat'

# XGB

In [225]:
folder_path = r"../data_sample/csv_output/"   # ปรับเป็น path ที่แท้จริง
patterns = ["*_lateral_*.csv", "*any*.csv"]
n_trials=5
df = load_csvs(folder_path, patterns)
subject_1 = df[df["frame"].str.contains("S01")]
subject_2 = df[df["frame"].str.contains("S02")]
subject_3 = df[df["frame"].str.contains("S03")]
subject_4 = df[df["frame"].str.contains("S04")]
subject_5 = df[df["frame"].str.contains("S05")]
subject_6 = df[df["frame"].str.contains("S06")]
X, y, df = preprocess_and_features(df)
print("Feature shape:", X.shape)

Combined shape: (8459, 101)output/S04_any_20_incorrect_incorrect_free_chest_normal_01_landmarks.csv shape: (116, 101)112, 101) 101))
clean_missing_coords: dropped 11 rows with missing x/y coords (from 8459 -> 8448)
Feature shape: (8448, 134)


In [226]:
# inspect class distribution
print("Class distribution before tuning:", dict(Counter(y)))

# standardize
scaler = StandardScaler()
Xs = scaler.fit_transform(X)


classes = np.unique(y)
class_w = compute_class_weight('balanced', classes=classes, y=y)
sample_weight = np.array([class_w[cls] for cls in y])

Class distribution before tuning: {np.int64(1): 4780, np.int64(0): 3668}


In [227]:
study = optuna.create_study(direction='maximize', study_name='xgb_study')
func = lambda trial: objective_xgb(trial, Xs, y, requested_folds=4)
study.optimize(func, n_trials=n_trials, show_progress_bar=True)
completed = [t for t in study.trials]
if len(completed) == 0:
    raise ValueError("No completed Optuna trials. Check your data, CV folds, or imputation strategy.")
print("Best XGB trial:", study.best_trial.params, "AUC:", study.best_value)
best_params = study.best_trial.params
if USE_GPU:
    best_params.update({'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor', 'gpu_id': 0})
else:
    best_params.update({'tree_method': 'hist'})

[I 2025-10-04 13:08:09,407] A new study created in memory with name: xgb_study


  0%|          | 0/5 [00:00<?, ?it/s]

prepare: X shape=(8448, 134), total NaNs=0
CV: class_counts={np.int64(1): 4780, np.int64(0): 3668}, using n_splits=4
objective_xgb: got NaN in CV scores: [nan nan nan nan]
[I 2025-10-04 13:08:13,881] Trial 0 finished with value: 0.0 and parameters: {'n_estimators': 1755, 'max_depth': 3, 'learning_rate': 0.00010447593614231826, 'subsample': 0.7255567255728854, 'colsample_bytree': 0.5512804483152499, 'reg_lambda': 0.0669299611695682, 'reg_alpha': 0.0100919130137838}. Best is trial 0 with value: 0.0.
prepare: X shape=(8448, 134), total NaNs=0
CV: class_counts={np.int64(1): 4780, np.int64(0): 3668}, using n_splits=4
objective_xgb: got NaN in CV scores: [nan nan nan nan]
[I 2025-10-04 13:08:32,261] Trial 1 finished with value: 0.0 and parameters: {'n_estimators': 1475, 'max_depth': 12, 'learning_rate': 0.00010744268630341623, 'subsample': 0.5545775042535113, 'colsample_bytree': 0.9093913416639042, 'reg_lambda': 0.007110811317572078, 'reg_alpha': 0.9368422731569387}. Best is trial 0 with val

In [228]:
best_params.update({'use_label_encoder': False, 'random_state': RANDOM_SEED})
best_clf = xgb.XGBClassifier(**best_params)

auto = False
if auto: 
    X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(Xs, y, sample_weight, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
else:
    auto = True
    if auto:
        df = pd.concat([subject_1, subject_2, subject_3, subject_4, subject_5])
        Xs, y, df = preprocess_and_features(df)
        sample_weight = np.array([class_w[cls] for cls in y])
        X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(Xs, y, sample_weight, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
    else:
        df = pd.concat([subject_2, subject_3, subject_5])
        X_tr, y_tr, df = preprocess_and_features(df)
        w_tr = np.array([class_w[cls] for cls in y_tr])
        df = pd.concat([subject_1])
        X_val, y_val, df = preprocess_and_features(df)
        w_val = np.array([class_w[cls] for cls in y_val])
        

    

try:
    best_clf.fit(X_tr, y_tr, sample_weight=w_tr,
                 eval_set=[(X_val, y_val)],
                 eval_metric='logloss',
                 early_stopping_rounds=25,
                 verbose=50)
except Exception as e:
    print("Warning: XGBoost training with GPU failed or other error, retrying with CPU. Error:", e)
    best_clf.set_params(tree_method='hist', predictor='cpu_predictor')
    best_clf.fit(X_tr, y_tr, sample_weight=w_tr,
                 eval_set=[(X_val, y_val)],
                 eval_metric='logloss',
                 early_stopping_rounds=50,
                 verbose=50)
# evaluate
if hasattr(best_clf, "predict_proba"):
    probs = best_clf.predict_proba(X_val)
else:
    # fallback to xgboost Booster predict
    try:
        dval = xgb.DMatrix(X_val)
        probs = best_clf.predict(dval)
    except Exception:
        raise RuntimeError("Model does not support predict_proba and is not an xgboost.Booster.")

n_classes = len(np.unique(y))
if n_classes == 2:
    if probs.ndim > 1:
        y_prob = probs[:,1]
    else:
        y_prob = probs
    y_pred = (y_prob >= 0.5).astype(int)
    auc = roc_auc_score(y_val, y_prob)
    ll = log_loss(y_val, y_prob)
else:
    y_prob = probs
    y_pred = probs.argmax(axis=1)
    auc = roc_auc_score(y_val, y_prob, multi_class="ovo")
    ll = log_loss(y_val, y_prob)

acc = accuracy_score(y_val, y_pred)
print("Validation AUC:", auc)
print("Validation Accuracy:", acc)
print("Validation LogLoss:", ll)
print("Classification Report:\n", classification_report(y_val, y_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))


clean_missing_coords: dropped 10 rows with missing x/y coords (from 7138 -> 7128)
[0]	validation_0-logloss:0.69310
[50]	validation_0-logloss:0.69113
[100]	validation_0-logloss:0.68914
[150]	validation_0-logloss:0.68718
[200]	validation_0-logloss:0.68524
[250]	validation_0-logloss:0.68331
[300]	validation_0-logloss:0.68138
[350]	validation_0-logloss:0.67947
[400]	validation_0-logloss:0.67761
[450]	validation_0-logloss:0.67575
[500]	validation_0-logloss:0.67389
[550]	validation_0-logloss:0.67204
[600]	validation_0-logloss:0.67022
[650]	validation_0-logloss:0.66840
[700]	validation_0-logloss:0.66658
[750]	validation_0-logloss:0.66477
[800]	validation_0-logloss:0.66299
[850]	validation_0-logloss:0.66122
[900]	validation_0-logloss:0.65947
[950]	validation_0-logloss:0.65773
[1000]	validation_0-logloss:0.65600
[1050]	validation_0-logloss:0.65428
[1100]	validation_0-logloss:0.65257
[1150]	validation_0-logloss:0.65089
[1200]	validation_0-logloss:0.64917
[1250]	validation_0-logloss:0.64748
[1300

In [235]:
folder_path = r"../data_sample/csv_output/"   # ปรับเป็น path ที่แท้จริง
patterns = ["*_lateral_*.csv", "*_any_*"]
n_trials=5
df = load_all_csvs(folder_path, patterns)
subject_6 = df[df["frame"].str.contains("S06")]
X, y, df = preprocess_and_features(subject_6)
# X, y, df = preprocess_and_features(df)
print("Feature shape:", X.shape)

# inspect class distribution
print("Class distribution before tuning:", dict(Counter(y)))

# standardize
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

X_val = Xs
y_val = y

# evaluate
if hasattr(best_clf, "predict_proba"):
    probs = best_clf.predict_proba(X_val)
else:
    # fallback to xgboost Booster predict
    try:
        dval = xgb.DMatrix(X_val)
        probs = best_clf.predict(dval)
    except Exception:
        raise RuntimeError("Model does not support predict_proba and is not an xgboost.Booster.")

n_classes = len(np.unique(y))
if n_classes == 2:
    if probs.ndim > 1:
        y_prob = probs[:,1]
    else:
        y_prob = probs
    y_pred = (y_prob >= 0.5).astype(int)
    auc = roc_auc_score(y_val, y_prob)
    ll = log_loss(y_val, y_prob)
else:
    y_prob = probs
    y_pred = probs.argmax(axis=1)
    auc = roc_auc_score(y_val, y_prob, multi_class="ovo")
    ll = log_loss(y_val, y_prob)

acc = accuracy_score(y_val, y_pred)
print("Prediction AUC:", auc)
print("Prediction Accuracy:", acc)
print("Prediction LogLoss:", ll)
print("Classification Report:\n", classification_report(y_val, y_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))


Combined shape: (140499, 101)tput/S04_any_20_incorrect_incorrect_free_chest_normal_01_landmarks.csv shape: (1932, 101)855, 101) 101))
clean_missing_coords: dropped 3 rows with missing x/y coords (from 21953 -> 21950)
Feature shape: (21950, 134)
Class distribution before tuning: {np.int64(1): 10585, np.int64(0): 11365}
Prediction AUC: 0.6311844804414684
Prediction Accuracy: 0.5866059225512529
Prediction LogLoss: 0.6807230409030423
Classification Report:
               precision    recall  f1-score   support

           0     0.5577    0.9743    0.7094     11365
           1     0.8606    0.1703    0.2844     10585

    accuracy                         0.5866     21950
   macro avg     0.7092    0.5723    0.4969     21950
weighted avg     0.7038    0.5866    0.5044     21950

Confusion Matrix:
 [[11073   292]
 [ 8782  1803]]


## XGB Prediction

# LGB

In [197]:
folder_path = r"../video_splitter/csv_output_new_nahee/"   # ปรับเป็น path ที่แท้จริง
patterns = ["*_lateral_raise_*.csv"]
n_trials=5
df = load_csvs(folder_path, patterns)
subject_1 = df[df["frame"].str.contains("S01")]
subject_2 = df[df["frame"].str.contains("S02")]
subject_3 = df[df["frame"].str.contains("S03")]
subject_5 = df[df["frame"].str.contains("S05")]
subject_6 = df[df["frame"].str.contains("S06")]
X, y, df = preprocess_and_features(df)
print("Feature shape:", X.shape)

Combined shape: (10917, 101)v_output_new_nahee/S02_1_lateral_raise_10_incorrect_front_floor_Plain_Dime_4.mp4_landmarks.csv shape: (58, 101)1) 101)1))
clean_missing_coords: dropped 223 rows with missing x/y coords (from 10917 -> 10694)
Feature shape: (10694, 134)


In [198]:
# inspect class distribution
print("Class distribution before tuning:", dict(Counter(y)))

# standardize
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

n_trials = 10
classes = np.unique(y)
class_w = compute_class_weight('balanced', classes=classes, y=y)
sample_weight = np.array([class_w[cls] for cls in y])

study = optuna.create_study(direction='maximize', study_name='lgb_study')
func = lambda trial: objective_lgb(trial, Xs, y, requested_folds=4)
study.optimize(func, n_trials=n_trials, show_progress_bar=True)
completed = [t for t in study.trials if t.state]
if len(completed) == 0:
    raise ValueError("No completed Optuna trials. Check your data, CV folds, or imputation strategy.")
print("Best LGB trial:", study.best_trial.params, "AUC:", study.best_value)
best_params = study.best_trial.params
if USE_GPU:
    best_params.update({'device': 'gpu'})


Class distribution before tuning: {np.int64(0): 4109, np.int64(1): 6585}


[I 2025-10-03 23:29:41,509] A new study created in memory with name: lgb_study


  0%|          | 0/10 [00:00<?, ?it/s]

prepare: X shape=(10694, 134), total NaNs=0
CV: class_counts={np.int64(0): 4109, np.int64(1): 6585}, using n_splits=4
[LightGBM] [Info] Number of positive: 4939, number of negative: 3081
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 33923
[LightGBM] [Info] Number of data points in the train set: 8020, number of used features: 134
[LightGBM] [Info] Number of positive: 4938, number of negative: 3082
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 33923
[LightGBM] [Info] Number of data points in the train set: 8020, number of used features: 134
[LightGBM] [Info] Number of positive: 4939, number of negative: 3082
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 33923
[LightGBM] [Info] Number of data points in the train set: 8021, number of used features: 134
[LightGBM] [Info] Number of positive: 4939, number of negative: 3082
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 33923
[LightGBM] 

In [199]:
best_params.update({'random_state': RANDOM_SEED})
best_clf = lgb.LGBMClassifier(**best_params)
auto = False
if auto: 
    X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(Xs, y, sample_weight, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
else:
    auto = True
    if auto:
        df = pd.concat([subject_1, subject_2, subject_3, subject_5])
        Xs, y, df = preprocess_and_features(df)
        sample_weight = np.array([class_w[cls] for cls in y])
        X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(Xs, y, sample_weight, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
    else:
        df = pd.concat([subject_2, subject_3, subject_5])
        X_tr, y_tr, df = preprocess_and_features(df)
        w_tr = np.array([class_w[cls] for cls in y_tr])
        df = pd.concat([subject_1])
        X_val, y_val, df = preprocess_and_features(df)
        w_val = np.array([class_w[cls] for cls in y_val])

try:
    best_clf.fit(X_tr, y_tr, sample_weight=w_tr,
                 eval_set=[(X_val, y_val)],
                 eval_metric='binary_logloss',
                 early_stopping_rounds=50,
                 verbose=50)
except Exception as e:
    print("Warning: LightGBM training failed with GPU flags, retrying CPU. Error:", e)
    params = best_clf.get_params()
    for k in ['device','gpu_platform_id','gpu_device_id']:
        if k in params: params.pop(k, None)
    best_clf = lgb.LGBMClassifier(**params)
    best_clf.fit(X_tr, y_tr, sample_weight=w_tr,
                 eval_set=[(X_val, y_val)],
                 eval_metric='binary_logloss'
                )
# evaluate
if hasattr(best_clf, "predict_proba"):
    probs = best_clf.predict_proba(X_val)
else:
    # fallback to xgboost Booster predict
    try:
        dval = xgb.DMatrix(X_val)
        probs = best_clf.predict(dval)
    except Exception:
        raise RuntimeError("Model does not support predict_proba and is not an xgboost.Booster.")

n_classes = len(np.unique(y))
if n_classes == 2:
    if probs.ndim > 1:
        y_prob = probs[:,1]
    else:
        y_prob = probs
    y_pred = (y_prob >= 0.5).astype(int)
    auc = roc_auc_score(y_val, y_prob)
    ll = log_loss(y_val, y_prob)
else:
    y_prob = probs
    y_pred = probs.argmax(axis=1)
    auc = roc_auc_score(y_val, y_prob, multi_class="ovo")
    ll = log_loss(y_val, y_prob)

acc = accuracy_score(y_val, y_pred)
print("Validation AUC:", auc)
print("Validation Accuracy:", acc)
print("Validation LogLoss:", ll)
print("Classification Report:\n", classification_report(y_val, y_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))


clean_missing_coords: dropped 105 rows with missing x/y coords (from 7924 -> 7819)
[LightGBM] [Info] Number of positive: 3905, number of negative: 2350
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001484 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 33920
[LightGBM] [Info] Number of data points in the train set: 6255, number of used features: 134
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.509056 -> initscore=0.036228
[LightGBM] [Info] Start training from score 0.036228
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positiv

In [200]:
folder_path = r"../video_splitter/csv_output_new_nahee/"   # ปรับเป็น path ที่แท้จริง
patterns = ["*_lateral_raise_*.csv"]
n_trials=5
df = load_all_csvs(folder_path, patterns)
subject_6 = df[df["frame"].str.contains("S06")]
X, y, df = preprocess_and_features(subject_6)
print("Feature shape:", X.shape)

# inspect class distribution
print("Class distribution before tuning:", dict(Counter(y)))

# standardize
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

X_val = Xs
y_val = y

# evaluate
if hasattr(best_clf, "predict_proba"):
    probs = best_clf.predict_proba(X_val)
else:
    # fallback to xgboost Booster predict
    try:
        dval = xgb.DMatrix(X_val)
        probs = best_clf.predict(dval)
    except Exception:
        raise RuntimeError("Model does not support predict_proba and is not an xgboost.Booster.")

n_classes = len(np.unique(y))
if n_classes == 2:
    if probs.ndim > 1:
        y_prob = probs[:,1]
    else:
        y_prob = probs
    y_pred = (y_prob >= 0.5).astype(int)
    auc = roc_auc_score(y_val, y_prob)
    ll = log_loss(y_val, y_prob)
else:
    y_prob = probs
    y_pred = probs.argmax(axis=1)
    auc = roc_auc_score(y_val, y_prob, multi_class="ovo")
    ll = log_loss(y_val, y_prob)

acc = accuracy_score(y_val, y_pred)
print("Predicting AUC:", auc)
print("Predicting Accuracy:", acc)
print("Predicting LogLoss:", ll)
print("Classification Report:\n", classification_report(y_val, y_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))


Combined shape: (181202, 101)_output_new_nahee/S02_1_lateral_raise_10_incorrect_front_floor_Plain_Dime_4.mp4_landmarks.csv shape: (967, 101)01)101)1))
clean_missing_coords: dropped 2072 rows with missing x/y coords (from 49739 -> 47667)
Feature shape: (47667, 134)
Class distribution before tuning: {np.int64(0): 19524, np.int64(1): 28143}
Predicting AUC: 0.5175591379854211
Predicting Accuracy: 0.4903602072712778
Predicting LogLoss: 2.160457403461305
Classification Report:
               precision    recall  f1-score   support

           0     0.4376    0.8570    0.5794     19524
           1     0.7041    0.2360    0.3535     28143

    accuracy                         0.4904     47667
   macro avg     0.5709    0.5465    0.4664     47667
weighted avg     0.5950    0.4904    0.4460     47667

Confusion Matrix:
 [[16733  2791]
 [21502  6641]]


## LGB Prediction

# CAT

In [201]:
folder_path = r"../video_splitter/csv_output_new_nahee/"   # ปรับเป็น path ที่แท้จริง
patterns = ["*_lateral_raise_*.csv"]
n_trials=5
df = load_csvs(folder_path, patterns)
subject_1 = df[df["frame"].str.contains("S01")]
subject_2 = df[df["frame"].str.contains("S02")]
subject_3 = df[df["frame"].str.contains("S03")]
subject_5 = df[df["frame"].str.contains("S05")]
subject_6 = df[df["frame"].str.contains("S06")]
X, y, df = preprocess_and_features(df)
print("Feature shape:", X.shape)

Combined shape: (10917, 101)v_output_new_nahee/S02_1_lateral_raise_10_incorrect_front_floor_Plain_Dime_4.mp4_landmarks.csv shape: (58, 101)1) 101)1))
clean_missing_coords: dropped 223 rows with missing x/y coords (from 10917 -> 10694)
Feature shape: (10694, 134)


In [202]:
# inspect class distribution
print("Class distribution before tuning:", dict(Counter(y)))

# standardize
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

classes = np.unique(y)
class_w = compute_class_weight('balanced', classes=classes, y=y)
sample_weight = np.array([class_w[cls] for cls in y])

if USE_GPU:
    best_clf = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, task_type='GPU', verbose=0, random_state=RANDOM_SEED)
else:
    best_clf = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, task_type='CPU', verbose=0, random_state=RANDOM_SEED)

Class distribution before tuning: {np.int64(0): 4109, np.int64(1): 6585}


In [203]:
auto = False
if auto: 
    X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(Xs, y, sample_weight, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
else:
    auto = True
    if auto:
        df = pd.concat([subject_1, subject_2, subject_3, subject_5])
        Xs, y, df = preprocess_and_features(df)
        sample_weight = np.array([class_w[cls] for cls in y])
        X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(Xs, y, sample_weight, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
    else:
        df = pd.concat([subject_2, subject_3, subject_5])
        X_tr, y_tr, df = preprocess_and_features(df)
        w_tr = np.array([class_w[cls] for cls in y_tr])
        df = pd.concat([subject_1])
        X_val, y_val, df = preprocess_and_features(df)
        w_val = np.array([class_w[cls] for cls in y_val])


best_clf.fit(X_tr, y_tr, sample_weight=w_tr)
# evaluate
if hasattr(best_clf, "predict_proba"):
    probs = best_clf.predict_proba(X_val)
else:
    # fallback to xgboost Booster predict
    try:
        dval = xgb.DMatrix(X_val)
        probs = best_clf.predict(dval)
    except Exception:
        raise RuntimeError("Model does not support predict_proba and is not an xgboost.Booster.")

n_classes = len(np.unique(y))
if n_classes == 2:
    if probs.ndim > 1:
        y_prob = probs[:,1]
    else:
        y_prob = probs
    y_pred = (y_prob >= 0.5).astype(int)
    auc = roc_auc_score(y_val, y_prob)
    ll = log_loss(y_val, y_prob)
else:
    y_prob = probs
    y_pred = probs.argmax(axis=1)
    auc = roc_auc_score(y_val, y_prob, multi_class="ovo")
    ll = log_loss(y_val, y_prob)

acc = accuracy_score(y_val, y_pred)
print("Validation AUC:", auc)
print("Validation Accuracy:", acc)
print("Validation LogLoss:", ll)
print("Classification Report:\n", classification_report(y_val, y_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))


clean_missing_coords: dropped 105 rows with missing x/y coords (from 7924 -> 7819)
Validation AUC: 0.9891647587877224
Validation Accuracy: 0.9501278772378516
Validation LogLoss: 0.14741734894081987
Classification Report:
               precision    recall  f1-score   support

           0     0.9504    0.9148    0.9323       587
           1     0.9499    0.9713    0.9605       977

    accuracy                         0.9501      1564
   macro avg     0.9502    0.9431    0.9464      1564
weighted avg     0.9501    0.9501    0.9499      1564

Confusion Matrix:
 [[537  50]
 [ 28 949]]


In [204]:
folder_path = r"../video_splitter/csv_output_new_nahee/"   # ปรับเป็น path ที่แท้จริง
patterns = ["*_lateral_raise_*.csv"]
n_trials=5
df = load_all_csvs(folder_path, patterns)
subject_6 = df[df["frame"].str.contains("S06")]
X, y, df = preprocess_and_features(subject_6)
print("Feature shape:", X.shape)

# inspect class distribution
print("Class distribution before tuning:", dict(Counter(y)))

# standardize
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

X_val = Xs
y_val = y

# evaluate
if hasattr(best_clf, "predict_proba"):
    probs = best_clf.predict_proba(X_val)
else:
    # fallback to xgboost Booster predict
    try:
        dval = xgb.DMatrix(X_val)
        probs = best_clf.predict(dval)
    except Exception:
        raise RuntimeError("Model does not support predict_proba and is not an xgboost.Booster.")

n_classes = len(np.unique(y))
if n_classes == 2:
    if probs.ndim > 1:
        y_prob = probs[:,1]
    else:
        y_prob = probs
    y_pred = (y_prob >= 0.5).astype(int)
    auc = roc_auc_score(y_val, y_prob)
    ll = log_loss(y_val, y_prob)
else:
    y_prob = probs
    y_pred = probs.argmax(axis=1)
    auc = roc_auc_score(y_val, y_prob, multi_class="ovo")
    ll = log_loss(y_val, y_prob)

acc = accuracy_score(y_val, y_pred)
print("Validation AUC:", auc)
print("Validation Accuracy:", acc)
print("Validation LogLoss:", ll)
print("Classification Report:\n", classification_report(y_val, y_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))


Combined shape: (181202, 101)_output_new_nahee/S02_1_lateral_raise_10_incorrect_front_floor_Plain_Dime_4.mp4_landmarks.csv shape: (967, 101)01)101)1))
clean_missing_coords: dropped 2072 rows with missing x/y coords (from 49739 -> 47667)
Feature shape: (47667, 134)
Class distribution before tuning: {np.int64(0): 19524, np.int64(1): 28143}
Validation AUC: 0.5126856643249151
Validation Accuracy: 0.4450458388402878
Validation LogLoss: 1.6163854818143797
Classification Report:
               precision    recall  f1-score   support

           0     0.4123    0.8338    0.5517     19524
           1     0.6033    0.1753    0.2717     28143

    accuracy                         0.4450     47667
   macro avg     0.5078    0.5046    0.4117     47667
weighted avg     0.5251    0.4450    0.3864     47667

Confusion Matrix:
 [[16280  3244]
 [23209  4934]]


## Cat Prediction